# Validación y optimización del agente MCTS para Connect-4

## 1. Introducción y diseño experimental

Connect-4 es un juego de Markov alternante, determinista, de suma cero y con información perfecta. En cada estado el jugador activo elige una columna legal, la transición coloca una ficha y la recompensa solo se determina al alcanzar victoria o empate. El espacio de secuencias legales hace que evaluar exhaustivamente todas las ramas sea costoso; por ello se utiliza **Monte-Carlo Tree Search (MCTS)** con UCB1 para asignar el recurso de cómputo a las acciones más prometedoras.

Se estudian tres políticas del repositorio:

- **MCTS base**: `groups/Fermin_MCTS/policy.py`, con UCB1, presupuesto por turno y rollout aleatorio.
- **MCTS mejorado**: `groups/Fermin_MCTS_Time&C/policy.py`, con cálculo adaptativo del presupuesto a partir de fase, ramificación e incertidumbre, más rollout que captura victorias inmediatas y declara una intención de bloqueo que será auditada experimentalmente.
- **Random**: `groups/Random/policy.py`, control que selecciona una columna válida al azar.

La variable numérica de recursos es `TURN_TIME_LIMIT` en $[0.10, 0.20, 0.35, 0.50]$ segundos. La corrida experimental incluida ejecuta **10 partidas reales por cada punto y emparejamiento**, alternando el jugador inicial; amplía la muestra piloto previa y mantiene viable el coste de MCTS, que crece rápidamente al aumentar el presupuesto. Se evalúan cuatro emparejamientos: `MCTS base vs Random`, `MCTS mejorado vs Random`, `MCTS base vs MCTS mejorado` y `MCTS mejorado vs MCTS mejorado` (self-play como control de simetría). En total se juegan $4 \times 4 \times 10 = 160$ partidas reales; el mismo runner permite ampliar presupuestos o tamaño muestral en una corrida posterior.

> **Trazabilidad técnica:** el código actual del agente mejorado demuestra presupuesto adaptativo y detección de victoria inmediata en el rollout. Conserva `C = sqrt(2)` constante y construye una raíz nueva en cada `act()`; además, la comprobación de bloqueo se audita abajo porque no simula explícitamente la jugada rival. Tabla de transposición y reúso de árbol se presentan como mejoras futuras, no como funcionalidades ya medidas.


In [ ]:
# 2. Configuración, imports y carga dinámica de políticas
from pathlib import Path
import importlib.util
import json
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import seaborn as sns
from IPython.display import display

from connect4.connect_state import ConnectState
from connect4.policy import Policy

sns.set_theme(style="whitegrid", context="talk")
TURN_LIMITS = [0.10, 0.20, 0.35, 0.50]
N_GAMES = 10  # Tamaño de la corrida real externa almacenada en CSV.
RUN_FULL_BENCHMARK = False  # El CSV se genero fuera del notebook con benchmark_entrega.py.
RESULTS_PATH = Path("versus/benchmark_entrega.csv")

POLICY_PATHS = {
    "Fermin_MCTS": Path("groups/Fermin_MCTS/policy.py"),
    "Fermin_MCTS_Time&C": Path("groups/Fermin_MCTS_Time&C/policy.py"),
    "Random": Path("groups/Random/policy.py"),
}


def load_policy_class(label: str, path: Path) -> type[Policy]:
    """Importa una política desde su ruta, incluso cuando contiene '&'."""
    module_name = f"entrega_{label.replace('&', 'and')}"
    spec = importlib.util.spec_from_file_location(module_name, path)
    if spec is None or spec.loader is None:
        raise ImportError(f"No fue posible importar {path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    candidates = [
        cls for cls in vars(module).values()
        if isinstance(cls, type)
        and issubclass(cls, Policy)
        and cls is not Policy
        and cls.__module__ == module.__name__
    ]
    if len(candidates) != 1:
        raise ValueError(f"Se esperaba una única política concreta en {path}: {candidates}")
    return candidates[0]


POLICIES = {name: load_policy_class(name, path) for name, path in POLICY_PATHS.items()}
pd.DataFrame({"agente": POLICY_PATHS.keys(), "archivo": [str(path) for path in POLICY_PATHS.values()]})


In [ ]:
# 3. Motor de evaluación real: partidas balanceadas y presupuesto configurable
MATCHUPS = [
    ("Base vs Random", "Fermin_MCTS", "Random"),
    ("Mejorado vs Random", "Fermin_MCTS_Time&C", "Random"),
    ("Base vs Mejorado", "Fermin_MCTS", "Fermin_MCTS_Time&C"),
    ("Mejorado vs Mejorado", "Fermin_MCTS_Time&C", "Fermin_MCTS_Time&C"),
]


def build_agent(policy_name: str, turn_limit: float) -> Policy:
    """Instancia una política fijando el recurso estudiado en ambas variantes MCTS."""
    agent = POLICIES[policy_name]()
    if hasattr(agent, "TURN_TIME_LIMIT"):
        agent.TURN_TIME_LIMIT = turn_limit
    try:
        agent.mount(turn_limit)
    except TypeError:
        agent.mount()
    return agent


def play_one_game(policy_a: str, policy_b: str, turn_limit: float, a_starts: bool, seed: int) -> dict:
    """Ejecuta una partida y mide tiempo de decisión de cada política."""
    random.seed(seed)
    agent_a = build_agent(policy_a, turn_limit)
    agent_b = build_agent(policy_b, turn_limit)
    first, second = (agent_a, agent_b) if a_starts else (agent_b, agent_a)
    state = ConnectState()
    elapsed = {"a": 0.0, "b": 0.0}
    moves = {"a": 0, "b": 0}

    while not state.is_final():
        is_role_a = (state.player == -1 and a_starts) or (state.player == 1 and not a_starts)
        current_role = "a" if is_role_a else "b"
        current = agent_a if is_role_a else agent_b
        start = time.perf_counter()
        action = int(current.act(state.board))
        elapsed[current_role] += time.perf_counter() - start
        moves[current_role] += 1
        state = state.transition(action)

    winner_token = state.get_winner()
    if winner_token == 0:
        winner_role = None
    else:
        first_won = winner_token == -1
        winner_role = "a" if first_won == a_starts else "b"
    return {
        "winner_role": winner_role,
        "elapsed_a": elapsed["a"],
        "elapsed_b": elapsed["b"],
        "moves_a": moves["a"],
        "moves_b": moves["b"],
    }


def evaluate_matchup(label: str, policy_a: str, policy_b: str, turn_limit: float, n_games: int = N_GAMES) -> dict:
    rows = [
        play_one_game(policy_a, policy_b, turn_limit, a_starts=(i % 2 == 0), seed=1200 + i)
        for i in range(n_games)
    ]
    wins_a = sum(row["winner_role"] == "a" for row in rows)
    wins_b = sum(row["winner_role"] == "b" for row in rows)
    draws = sum(row["winner_role"] is None for row in rows)
    total_moves_a = sum(row["moves_a"] for row in rows)
    total_moves_b = sum(row["moves_b"] for row in rows)
    return {
        "matchup": label,
        "agent_a": policy_a,
        "agent_b": policy_b,
        "time_limit_per_turn": turn_limit,
        "games": n_games,
        "wins_a": wins_a,
        "wins_b": wins_b,
        "draws": draws,
        "win_rate_a": wins_a / n_games,
        "win_rate_b": wins_b / n_games,
        "draw_rate": draws / n_games,
        "avg_move_ms_a": 1000 * sum(row["elapsed_a"] for row in rows) / max(1, total_moves_a),
        "avg_move_ms_b": 1000 * sum(row["elapsed_b"] for row in rows) / max(1, total_moves_b),
        "source": "benchmark ejecutado",
    }


def run_full_benchmark() -> pd.DataFrame:
    records = []
    for label, policy_a, policy_b in MATCHUPS:
        for turn_limit in TURN_LIMITS:
            records.append(evaluate_matchup(label, policy_a, policy_b, turn_limit))
    result = pd.DataFrame(records)
    RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(RESULTS_PATH, index=False)
    return result


In [ ]:
# 4. Datos para el informe: solo resultados de partidas reales exportados a CSV
def cached_results_are_complete(cached: pd.DataFrame) -> bool:
    expected = {(label, budget) for label, _, _ in MATCHUPS for budget in TURN_LIMITS}
    observed = set(zip(cached["matchup"], cached["time_limit_per_turn"].round(2)))
    return expected == observed


if RUN_FULL_BENCHMARK:
    results = run_full_benchmark()
elif RESULTS_PATH.exists():
    results = pd.read_csv(RESULTS_PATH)
else:
    raise FileNotFoundError("Ejecute benchmark_entrega.py antes de renderizar el informe.")

if not cached_results_are_complete(results):
    raise ValueError("El CSV real no contiene los cuatro emparejamientos en los presupuestos definidos.")

print("Fuente utilizada:", results["source"].iloc[0])
results.sort_values(["matchup", "time_limit_per_turn"]).reset_index(drop=True)


In [ ]:
# Evidencia piloto disponible: se informa como tabla y no sustituye el benchmark principal
existing_match_path = Path("versus/match_Fermin_MCTS_vs_Fermin_MCTS_Time&C.json")
if existing_match_path.exists():
    with existing_match_path.open("r", encoding="utf-8") as match_file:
        existing_match = json.load(match_file)
    existing_summary = pd.DataFrame({
        "resultado": ["Fermin_MCTS", "Fermin_MCTS_Time&C", "Empates"],
        "partidas": [existing_match["player_a_wins"], existing_match["player_b_wins"], existing_match["draws"]],
    })
    print("Evidencia piloto cargada (n=7); las figuras principales usan los CSV reales externos.")
    display(existing_summary)
else:
    print("No hay un match JSON previo entre las dos versiones.")


El archivo de torneo previamente disponible contiene siete partidas entre las dos variantes: el agente base ganó 2, el mejorado ganó 4 y se produjo 1 empate. Esta muestra se informa como **piloto tabular**, no como gráfica principal, porque es insuficiente para una conclusión estadística. Las figuras del informe se construyen con el CSV externo de partidas reales: 10 partidas por cada uno de cuatro presupuestos y cuatro emparejamientos.


In [ ]:
# Figura 1. Criterio 1: ambas versiones contra Random según el recurso numérico
against_random = results[results["matchup"].isin(["Base vs Random", "Mejorado vs Random"])].copy()
against_random["version"] = against_random["agent_a"]
palette_agents = {"Fermin_MCTS": "#31688e", "Fermin_MCTS_Time&C": "#35b779"}

fig, ax = plt.subplots(figsize=(14, 7.5), layout="constrained")
sns.lineplot(
    data=against_random, x="time_limit_per_turn", y="win_rate_a", hue="version",
    style="version", markers=True, dashes=False, linewidth=3, markersize=10,
    palette=palette_agents, ax=ax,
)
ax.axhline(0.85, color="firebrick", linestyle="--", linewidth=2, label="Criterio mínimo: 85%")
fig.suptitle("Figura 1. Tasa de victorias frente a Random", weight="bold", fontsize=19)
ax.set_title(f"Cada marcador resume {N_GAMES} partidas reales balanceadas por jugador inicial", fontsize=12, pad=12)
ax.set_xlabel("TURN_TIME_LIMIT (segundos)")
ax.set_ylabel("Tasa de victorias contra Random")
ax.set_xticks(TURN_LIMITS, [f"{budget:.2f}" for budget in TURN_LIMITS])
ax.set_ylim(0.82, 1.005)
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.legend(title="Política / umbral", loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3, frameon=True)
sns.despine()
plt.show()

summary_random = against_random.groupby("version", as_index=False)["win_rate_a"].mean()
summary_random["victorias_promedio"] = summary_random["win_rate_a"] * N_GAMES
fig, ax = plt.subplots(figsize=(11, 6.5), layout="constrained")
sns.barplot(data=summary_random, x="version", y="victorias_promedio", hue="version", palette=palette_agents, legend=False, ax=ax)
ax.axhline(0.85 * N_GAMES, color="firebrick", linestyle="--", linewidth=2)
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f", padding=4)
fig.suptitle("Figura 2. Victorias promedio frente a Random", weight="bold", fontsize=18)
ax.set_title(f"Promedio sobre los cuatro presupuestos; {N_GAMES} partidas reales por condición", fontsize=12, pad=12)
ax.set_xlabel("Versión")
ax.set_ylabel(f"Victorias promedio por cada {N_GAMES} partidas")
ax.set_ylim(0, N_GAMES + 0.5)
plt.show()


In [ ]:
# Figura 3. Comparación directa: MCTS base contra MCTS mejorado
head_to_head = results[results["matchup"] == "Base vs Mejorado"].sort_values("time_limit_per_turn").copy()
head_to_head["Base"] = head_to_head["win_rate_a"]
head_to_head["Mejorado"] = head_to_head["win_rate_b"]
head_to_head["Empate"] = head_to_head["draw_rate"]
stack_data = head_to_head.set_index("time_limit_per_turn")[["Base", "Mejorado", "Empate"]]

fig, ax = plt.subplots(figsize=(14, 7), layout="constrained")
stack_data.plot(
    kind="bar", stacked=True, ax=ax,
    color=["#31688e", "#35b779", "#d9d9d9"], width=0.72,
)
fig.suptitle("Figura 3. Enfrentamiento directo: base vs mejorado", weight="bold", fontsize=18)
ax.set_title(f"Distribución de resultados en {N_GAMES} partidas reales para cada presupuesto", fontsize=12, pad=12)
ax.set_xlabel("TURN_TIME_LIMIT (segundos)")
ax.set_ylabel("Proporción de partidas")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.legend(title="Resultado", loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3)
plt.xticks(rotation=0)
plt.show()


In [ ]:
# Figura 4. Control de estabilidad: agente mejorado contra sí mismo
self_play = results[results["matchup"] == "Mejorado vs Mejorado"].sort_values("time_limit_per_turn").copy()
self_play["Rol A"] = self_play["win_rate_a"]
self_play["Rol B"] = self_play["win_rate_b"]
self_play["Empate"] = self_play["draw_rate"]
self_stack = self_play.set_index("time_limit_per_turn")[["Rol A", "Rol B", "Empate"]]

fig, ax = plt.subplots(figsize=(14, 7), layout="constrained")
self_stack.plot(
    kind="bar", stacked=True, ax=ax,
    color=["#21918c", "#5ec962", "#d9d9d9"], width=0.72,
)
ax.axhline(0.5, color="black", linestyle="--", linewidth=1.5, label="Simetría esperada (50%)")
fig.suptitle("Figura 4. Self-play del agente mejorado", weight="bold", fontsize=18)
ax.set_title("Control de simetría: dos instancias idénticas con inicio balanceado", fontsize=12, pad=12)
ax.set_xlabel("TURN_TIME_LIMIT (segundos)")
ax.set_ylabel("Proporción de partidas")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.legend(title="Resultado", loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=4)
plt.xticks(rotation=0)
plt.show()


In [ ]:
# Figuras 5 y 6. Resumen global y frontera rendimiento-recurso
heatmap_table = results.pivot(index="matchup", columns="time_limit_per_turn", values="win_rate_a")
fig, ax = plt.subplots(figsize=(15, 6.5), layout="constrained")
sns.heatmap(
    heatmap_table, annot=True, fmt=".0%", cmap="viridis", vmin=0, vmax=1,
    linewidths=0.7, cbar_kws={"label": "Win rate del agente A"}, ax=ax,
)
fig.suptitle("Figura 5. Resumen de tasa de victorias del agente A", weight="bold", fontsize=18)
ax.set_title(f"Cada celda corresponde a {N_GAMES} partidas reales; A es la primera política del emparejamiento", fontsize=12, pad=12)
ax.set_xlabel("TURN_TIME_LIMIT (segundos)")
ax.set_ylabel("Emparejamiento (agente A vs agente B)")
plt.show()

# Frontera rendimiento-recurso para el criterio de optimización frente a Random.
fig, ax = plt.subplots(figsize=(14, 7), layout="constrained")
sns.scatterplot(
    data=against_random, x="avg_move_ms_a", y="win_rate_a", hue="version",
    style="time_limit_per_turn", palette=palette_agents, s=170, ax=ax,
)
fig.suptitle("Figura 6. Frontera rendimiento-recurso contra Random", weight="bold", fontsize=18)
ax.set_title("Arriba e izquierda representa mayor rendimiento con menor coste temporal", fontsize=12, pad=12)
ax.set_xlabel("Tiempo medio de decisión del agente A (ms)")
ax.set_ylabel("Tasa de victorias")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.legend(title="Versión / límite", loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=4)
sns.despine()
plt.show()


In [ ]:
# 10. Diagnóstico de alto impacto: la condición de bloqueo del rollout mejorado
threat_board = np.zeros((6, 7), dtype=int)
threat_board[5, 0:3] = 1  # Yellow ganaría colocando en la columna 3.
current_state = ConnectState(threat_board, player=-1)  # Turno de Red: debe bloquear.
legal_actions = current_state.get_free_cols()

# Lógica equivalente al bucle actualmente implementado en el rollout mejorado.
blocks_detected_by_current_code = []
for action in legal_actions:
    next_state = current_state.transition(action)  # Siempre coloca ficha roja.
    if next_state.is_final() and next_state.get_winner() == 1:
        blocks_detected_by_current_code.append(action)

# Diagnóstico correcto: simular el turno del rival para identificar su victoria inmediata.
opponent_state = ConnectState(current_state.board, player=1)
winning_threats = [
    action for action in opponent_state.get_free_cols()
    if opponent_state.transition(action).is_final()
    and opponent_state.transition(action).get_winner() == 1
]
blocking_diagnostic = pd.DataFrame({
    "métrica": ["Amenaza rival real", "Bloqueos detectados por el rollout actual"],
    "columnas": [winning_threats, blocks_detected_by_current_code],
})
blocking_diagnostic


In [ ]:
# 11. Auditoría automática de funcionalidades implementadas en las dos versiones
base_source = POLICY_PATHS["Fermin_MCTS"].read_text(encoding="utf-8")
improved_source = POLICY_PATHS["Fermin_MCTS_Time&C"].read_text(encoding="utf-8")
audit = pd.DataFrame([
    {"funcionalidad": "UCB1 en selección", "Base": "Sí", "Mejorado": "Sí"},
    {"funcionalidad": "Presupuesto adaptativo (_compute_budget)", "Base": "No", "Mejorado": "Sí" if "_compute_budget" in improved_source else "No"},
    {"funcionalidad": "Rollout: victoria inmediata", "Base": "No", "Mejorado": "Sí" if "forced" in improved_source else "No"},
    {"funcionalidad": "Rollout: bloqueo inmediato efectivo", "Base": "No", "Mejorado": "No: no simula la ficha rival"},
    {"funcionalidad": "C variable por profundidad", "Base": "No", "Mejorado": "Sí" if "dynamic_c" in improved_source else "No"},
    {"funcionalidad": "Tabla de transposición", "Base": "No", "Mejorado": "Sí" if "transposition" in improved_source.lower() else "No"},
    {"funcionalidad": "Reúso de árbol entre turnos", "Base": "No", "Mejorado": "Sí" if "self.root" in improved_source else "No"},
])
audit


## 12. Criterio 1 - Análisis de cada gráfica

### Figura 1. Tasa de victorias frente a Random

Esta es la gráfica principal para acreditar el criterio de desempeño. Cada marcador representa **10 partidas reales**, no una única partida, y se estudian cuatro presupuestos entre 0.10 s y 0.50 s. En los datos obtenidos, tanto `Fermin_MCTS` como `Fermin_MCTS_Time&C` alcanzan 100% de victorias en cada presupuesto: cada versión ganó `40/40` partidas contra `Random`, por encima del umbral del 85%. El resultado demuestra que ambos agentes satisfacen el control mínimo, pero también revela un efecto techo: el aleatorio es demasiado débil para medir cuál versión es mejor.

### Figura 2. Victorias promedio contra Random

Esta barra resume los cuatro presupuestos en una sola cifra por política. Ambos agentes obtienen un promedio de `10.0/10` victorias por condición frente a `Random`. Por ello, la barra confirma robustez contra el control, pero no demuestra aporte marginal del agente mejorado; para esa pregunta se necesita el enfrentamiento directo de la Figura 3.

### Figura 3. MCTS base contra MCTS mejorado

Esta comparación es más exigente que jugar contra `Random`, porque elimina el efecto techo. Los resultados reales fueron: a `0.10 s`, base `5`, mejorado `4`, empate `1`; a `0.20 s`, base `6-4`; a `0.35 s`, empate `5-5`; y a `0.50 s`, base `7-3`. En total, la versión base gana `23/40`, la mejorada `16/40` y hay `1` empate. Por tanto, esta corrida no demuestra una mejora competitiva del segundo agente: aunque su diseño intenta ahorrar recurso, requiere corrección y nueva evaluación antes de sustituir al original. El match previo (`4-2-1` favorable al mejorado) queda superado en tamaño por esta muestra y muestra la necesidad de múltiples partidas.

### Figura 4. Self-play del agente mejorado

En esta figura ambos jugadores ejecutan la misma política. Por tanto, no se busca una tasa de victoria alta, sino simetría: con el jugador inicial balanceado, los roles A y B deberían acercarse a 50%. Se observaron `15` victorias del rol A, `24` del rol B y `1` empate. Con solo diez juegos por presupuesto, este desequilibrio es una alerta exploratoria, no una prueba concluyente; motiva ampliar la muestra y controlar mejor la aleatoriedad antes de atribuir toda la ventaja head-to-head exclusivamente a cambios algorítmicos.

### Figura 5. Mapa de calor global

El mapa condensa todas las condiciones reales y colorea el `win_rate` del agente A. Las dos filas contra `Random` quedan saturadas en 100%; la fila `Base vs Mejorado` concentra la información discriminante, con ventaja del base salvo empate a `0.35 s`; y la fila de self-play expone la desviación de simetría que debe estudiarse en futuras repeticiones. El mapa muestra por qué un único oponente de control no basta para declarar optimización.

### Figura 6. Frontera rendimiento-recurso

Esta dispersión responde a la pregunta de optimización: no basta con ganar, también importa cuánto tiempo real medio se consume por decisión. Contra `Random`, ambos agentes se sitúan en 100% de victorias, pero el mejorado consume menos tiempo medio en todos los límites evaluados: aproximadamente `72 vs 76 ms` a `0.10 s`, `127 vs 161 ms` a `0.20 s`, `196 vs 280 ms` a `0.35 s` y `252 vs 392 ms` a `0.50 s`. Hay una mejora de eficiencia frente al control; sin embargo, la Figura 3 demuestra que este ahorro vino acompañado de menor rendimiento contra el MCTS base.

**Alcance de la evidencia.** Las figuras cargan exclusivamente `versus/benchmark_entrega.csv`, generado fuera del notebook ejecutando las políticas reales. Esta corrida de 160 partidas es una validación experimental exploratoria; una extensión con más juegos y presupuestos altos reduciría la incertidumbre y ampliaría la caracterización del coste.


## 13. Criterio 2 - Cuellos de botella y propuesta de mejora

### Mejoras verificadas del segundo agente

La auditoría del código muestra dos modificaciones presentes en `Fermin_MCTS_Time&C`, una de ellas parcialmente correcta:

1. **Presupuesto adaptativo.** El agente realiza una primera pasada de 0.08 s, calcula un presupuesto mediante `_compute_budget()` y considera fase del juego, factor de ramificación e incertidumbre observada en la raíz. Causa: el agente base gasta el mismo límite incluso en posiciones triviales o muy forzadas. Efecto observado contra `Random`: menor latencia media del mejorado en los cuatro presupuestos manteniendo `40/40` victorias. Límite observado contra un rival competente: la reducción de coste no preservó fuerza, pues el base ganó el head-to-head `23-16-1`.
2. **Rollout táctico parcial.** El rollout mejorado encuentra una victoria inmediata del jugador actual. Sin embargo, para bloquear compara el ganador rival después de aplicar una ficha del jugador actual; la prueba construida muestra una amenaza rival en la columna 3 que no es detectada. Causa: no se simula el estado espejo del oponente. Efecto: el supuesto escudo defensivo del rollout no aporta la mejora esperada y puede contaminar las estimaciones en posiciones tácticas.

La versión actual no implementa `C` decreciente por profundidad, tabla de transposición ni reúso del árbol. Afirmar que esas técnicas explican resultados del código actual sería incorrecto; son oportunidades de alto impacto precisamente porque el ahorro temporal actual no fue suficiente para mejorar el desempeño contra el agente base.

### Cuellos de botella y relación causa-efecto

- **Reinicio de la búsqueda en cada turno.** Ambos archivos construyen `root = Node(...)` dentro de `act()`. Causa: se pierden las visitas y valores acumulados en el subárbol que corresponde a la jugada realmente ejecutada. Efecto: se repite cómputo que ya había evaluado estados relevantes, reduciendo la calidad obtenida por segundo.
- **Estados repetidos evaluados de forma independiente.** El árbol no contiene una tabla indexada por `(tablero, jugador)`. Causa: secuencias de acciones distintas pueden llegar a estados equivalentes o tácticamente idénticos y volver a simularse. Efecto: más llamadas a `transition()`, `is_final()` y rollouts para información redundante.
- **Exploración uniforme en toda profundidad.** `EXPLORATION = sqrt(2)` se pasa sin adaptación a `ucb1_child()`. Causa: ramas profundas reciben la misma presión exploratoria que decisiones cercanas a la raíz. Efecto potencial: parte del presupuesto se utiliza en diversidad profunda que tiene menor impacto inmediato sobre la acción elegida.

### Mejoras concretas propuestas

1. **Corregir primero el bloqueo del rollout.** Construir `opponent_state = ConnectState(current.board, -current.player)`, localizar su acción ganadora inmediata y jugar esa columna desde `current` para bloquearla. La celda diagnóstica anterior se convierte en prueba de regresión mínima: la amenaza `[3]` debe producir bloqueo `[3]`.
2. **Reúso de árbol persistente.** Mantener `self.root` y, en el turno siguiente, localizar el hijo compatible con el tablero observado para promoverlo a raíz. Deben invalidarse ramas incompatibles. Métrica de validación: porcentaje de turnos con raíz reutilizada, visitas iniciales conservadas y `win_rate` por milisegundo.
3. **Tabla de transposición.** Almacenar estadísticas por una clave inmutable como `(board.tobytes(), player)` y compartir `visits/value` al reencontrar el estado. La consulta hash tiene coste esperado amortizado $O(1)$, aunque la búsqueda completa no se vuelve $O(1)$. Métrica: tasa de hits, nodos únicos, simulaciones evitadas y rendimiento contra base bajo 0.2 s.
4. **Exploración dependiente de profundidad.** Evaluar $C(d)=C_0/(1+\alpha d)$ con `alpha` en una rejilla pequeña. Métrica: tasa de victoria head-to-head y entropía de visitas en la raíz, evitando ajustar solo contra `Random`.
5. **Instrumentación del MCTS.** Registrar iteraciones, profundidad máxima/media, hits de transposición y tiempo por acción. Sin estas métricas, la gráfica rendimiento-recurso muestra el efecto externo pero no permite asignar causalidad interna con suficiente rigor.

Como extensión posterior, un libro de aperturas versionado y simétricamente canonicalizado permitiría resolver las primeras jugadas conocidas con consulta constante, reservando MCTS para el medio juego. Esta mejora debe probarse después del reúso y la tabla, porque ambos atacan un coste presente en todas las fases de la partida.
